# Tome 9, Chapter 1: Categories (Python companion)

Functors and natural transformations as programmers meet them, the free theorems that naturality gives, and the free monoid adjunction behind MapReduce.

*Before each solution, predict what the output will look like.*

## Exercises

**P1.1 (★) Functors in code.** A *functor* in a programming language is a type constructor `F` (lists, optional values, binary trees) with a function `fmap(f, x)` that applies `f : A → B` inside an `F A` to give an `F B`. Definition 1.3 becomes two *laws*: `fmap(id, x) == x` and `fmap(g∘f, x) == fmap(g, fmap(f, x))`. (a) Write `fmap` for lists, for optional values (`None` or a value) and for binary trees, and test both laws on thousands of random inputs and random functions. (b) Write a plausible-looking `fmap` for lists that breaks a law, and let the test find it.

**P1.2 (★) Natural transformations and *theorems for free*.** A function defined for lists of *any* element type, such as `reverse`, `safe_head : List A → Optional A` or `flatten : List (List A) → List A`, cannot look at the elements, only move them around. (a) Test the naturality square of Definition 1.7, `fmap(f, eta(x)) == eta(fmap(f, x))`, for these three. (b) Test it for `sorted`, which does look at the elements. For which functions `f` does the square still commute? This is Wadler's *Theorems for free!* (1989): naturality comes for free from the type.

**P1.3 (★★) CS application: the free monoid and MapReduce.** The list type is the free monoid: a map `h : A → M` into a monoid $(M,\cdot,e)$ extends uniquely to a monoid homomorphism `fold_h : List A → M`, `fold_h([a1,…,an]) = h(a1)·…·h(an)` (the adjunction of Example 1.21(a), for monoids). Because `fold_h` is a homomorphism, `fold_h(x + y) = fold_h(x) · fold_h(y)`, and a long list can be cut into chunks processed on different machines, the results being multiplied at the end: this is the idea behind MapReduce. (a) Compute word counts, the maximum, and the mean (with the monoid of pairs (sum, count)) of a large list in chunks, and check against a sequential computation. (b) Show experimentally that the median is not a monoid homomorphism of this kind, whatever monoid one tries to put on medians: find two pairs of lists with equal medians whose concatenations have different medians. (c) Matrix products are a monoid too: compute the $n$-th Fibonacci number as a fold of $2\times2$ matrices in parallel chunks.

## Solutions

### P1.1

In [ ]:
import random
random.seed(1)

def fmap_list(f, xs):
    return [f(x) for x in xs]

def fmap_opt(f, x):
    return None if x is None else f(x)

# binary trees: None (leaf) or a tuple (left, value, right)
def fmap_tree(f, t):
    return None if t is None else (fmap_tree(f, t[0]), f(t[1]), fmap_tree(f, t[2]))

def rand_list():
    return [random.randint(-50, 50) for _ in range(random.randint(0, 8))]

def rand_opt():
    return None if random.random() < 0.3 else random.randint(-50, 50)

def rand_tree(d=3):
    if d == 0 or random.random() < 0.3:
        return None
    return (rand_tree(d - 1), random.randint(-50, 50), rand_tree(d - 1))

def rand_fun():
    a, b, k = random.randint(-5, 5), random.randint(-5, 5), random.randint(2, 7)
    return random.choice([lambda x: a * x + b, lambda x: x * x - a, lambda x: x % k, lambda x: -x])

def check(fmap, gen, trials=5000):
    ident = all(fmap(lambda x: x, x) == x for x in (gen() for _ in range(trials)))
    comp = True
    for _ in range(trials):
        f, g, x = rand_fun(), rand_fun(), gen()
        comp &= fmap(lambda y: g(f(y)), x) == fmap(g, fmap(f, x))
    return ident, comp

for name, fm, gen in [("list", fmap_list, rand_list), ("optional", fmap_opt, rand_opt), ("tree", fmap_tree, rand_tree)]:
    print(f"{name:9s} identity law: {check(fm, gen)[0]},  composition law: {check(fm, gen)[1]}")

All three pass. Note that the laws are properties of `fmap` alone; nothing in Python enforces them, which is why they have to be tested (or proved).

In [ ]:
def fmap_bad(f, xs):
    """Applies f, then drops consecutive duplicates of the results: looks harmless."""
    out = []
    for y in (f(x) for x in xs):
        if not out or out[-1] != y:
            out.append(y)
    return out

print("bad fmap: identity law", check(fmap_bad, rand_list)[0], "  composition law", check(fmap_bad, rand_list)[1])
for _ in range(10000):
    f, g, x = rand_fun(), rand_fun(), rand_list()
    if fmap_bad(lambda y: g(f(y)), x) != fmap_bad(g, fmap_bad(f, x)):
        print("counterexample: x =", x, " f(x) =", [f(t) for t in x], " g(f(x)) =", [g(f(t)) for t in x])
        print("  fmap(g o f, x) =", fmap_bad(lambda y: g(f(y)), x), "  fmap(g, fmap(f, x)) =", fmap_bad(g, fmap_bad(f, x)))
        break

The identity law fails: a list with equal neighbours loses some of them, so `fmap(id, x) != x`. The composition law, on the other hand, holds, and no counterexample turns up: removing repeated neighbours before or after applying `g` gives the same result, because equal neighbours stay equal under `g`. So the two laws are independent, and a test of one says nothing about the other. A functor must not change the shape of the container, only its contents.

### P1.2

In [ ]:
def reverse(xs): return xs[::-1]
def safe_head(xs): return xs[0] if xs else None
def flatten(xss): return [x for xs in xss for x in xs]

def natural(eta, fmap_in, fmap_out, gen, trials=5000, funs=None):
    for _ in range(trials):
        f, x = (funs or rand_fun)(), gen()
        if fmap_out(f, eta(x)) != eta(fmap_in(f, x)):
            return False, x
    return True, None

nested = lambda: [rand_list() for _ in range(random.randint(0, 4))]
fmap_nested = lambda f, xss: [[f(x) for x in xs] for xs in xss]

print("reverse  :", natural(reverse, fmap_list, fmap_list, rand_list)[0])
print("safe_head:", natural(safe_head, fmap_list, fmap_opt, rand_list)[0])
print("flatten  :", natural(flatten, fmap_nested, fmap_list, nested)[0])
print("sorted   :", natural(sorted, fmap_list, fmap_list, rand_list))

The three polymorphic functions are natural; `sorted` is not, because it compares elements, and `f` may not preserve the order. It is natural for exactly the functions that keep comparisons (for the test below, the strictly increasing ones, and also the merely non-decreasing ones, since equal elements sort to equal values):

In [ ]:
increasing = lambda: random.choice([lambda x: 3 * x + 1, lambda x: x**3, lambda x: x // 4, lambda x: max(x, 0)])
decreasing = lambda: random.choice([lambda x: -x, lambda x: 10 - 2 * x])
print("sorted, non-decreasing f:", natural(sorted, fmap_list, fmap_list, rand_list, funs=increasing)[0])
print("sorted, decreasing f    :", natural(sorted, fmap_list, fmap_list, rand_list, funs=decreasing)[0])

So `sorted` is a natural transformation on the category of ordered sets and order-preserving maps, but not on all sets. Wadler's theorem says that any function written for an arbitrary type parameter, without inspecting values, satisfies the naturality square automatically: one gets a theorem from the type signature.

### P1.3

In [ ]:
import re, collections, statistics
from functools import reduce

words = re.findall(r"[a-z]+", ("""Many constructions of algebra take objects to other objects and maps to maps,
respecting composition; a functor is such a construction, and a natural transformation compares two of them.
Category theory was invented to say what natural means, and adjoint functors arise everywhere. """ * 4000).lower())
print(len(words), "words")

def mapreduce(xs, h, op, e, chunks=8):
    """fold_h computed on chunks (as on separate machines), the chunk results multiplied at the end."""
    n = len(xs)
    parts = [xs[i * n // chunks:(i + 1) * n // chunks] for i in range(chunks)]
    partial = [reduce(op, map(h, p), e) for p in parts]
    return reduce(op, partial, e)

# word count: the monoid of multisets (Counters) under addition
add_counts = lambda a, b: a + b
wc_par = mapreduce(words, lambda w: collections.Counter([w]), add_counts, collections.Counter(), chunks=8)
print("word count agrees:", wc_par == collections.Counter(words), "  most common:", wc_par.most_common(3))

nums = [random.gauss(0, 1) for _ in range(10**6)]
print("max agrees      :", mapreduce(nums, lambda x: x, max, float("-inf")) == max(nums))
s, c = mapreduce(nums, lambda x: (x, 1), lambda a, b: (a[0] + b[0], a[1] + b[1]), (0.0, 0))
print("mean: parallel", s / c, " sequential", statistics.fmean(nums))

Word counts and the maximum agree exactly; the means agree up to rounding, because floating-point addition is only approximately associative, so the 'monoid' of floats is a monoid only up to rounding. The mean itself is not a homomorphism (the mean of a concatenation is not a function of the two means), but it is the quotient of two homomorphisms, sum and count: that is the trick.

In [ ]:
med = statistics.median
x1, x2, z = [0, 2, 4], [2, 2, 2], [3, 3]
print("med([0,2,4]) =", med(x1), "= med([2,2,2]) =", med(x2), "  but after appending [3,3]:", med(x1 + z), "vs", med(x2 + z))

If the median of a concatenation were $m(x)\cdot m(y)$ for some operation $\cdot$, then lists with equal medians would give equal medians after appending the same list. The lists $[0,2,4]$ and $[2,2,2]$ both have median $2$, but after appending $[3,3]$ the medians are $3$ and $2$. So no monoid structure on medians can make the median a homomorphism, and computing a median in parallel needs more than one number per chunk (in practice: a sketch of the whole distribution).

In [ ]:
def matmul(A, B):
    return ((A[0][0] * B[0][0] + A[0][1] * B[1][0], A[0][0] * B[0][1] + A[0][1] * B[1][1]),
            (A[1][0] * B[0][0] + A[1][1] * B[1][0], A[1][0] * B[0][1] + A[1][1] * B[1][1]))

I2, Q = ((1, 0), (0, 1)), ((1, 1), (1, 0))
n = 10000
M = mapreduce([Q] * n, lambda m: m, matmul, I2, chunks=16)
a, b = 0, 1
for _ in range(n):
    a, b = b, a + b
print("F_10000 from 16 chunks equals the sequential one:", M[0][1] == a, "  it has", len(str(a)), "digits")

Matrix multiplication is associative but not commutative, and the chunks are multiplied in order: a monoid homomorphism needs associativity only. This is why parallel prefix sums, parsing with finite automata (products of transition matrices) and many other computations split so well across machines.